In [1]:
import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt
import engines as engs

from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path


eng = engs.get_engine()
text_qry = text(engs.load_qry('qry_olos.sql'))
df = pd.read_sql(text_qry, eng)


In [24]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [36]:
df

,area,base_type,tablename,data,hour,tentativas,atendidas,day_week,semana_mes
8331,Psicologia,inativa,_1605_PSICOLOGIA_INATIVA_0_18A25PT02_06112025_...,2025-11-07,12.0,243,7,Sexta_W1,1
5904,Medicina,Base Leads,_1605_MEDICINA_POS_0_LEADSPOSARTMEDPT21_051120...,2025-11-07,17.0,120,4,Sexta_W1,1
1020,Medicina,inativa,_1605_MEDICINA_INATIVA_0_18A25PT07_07112025_V498,2025-11-07,15.0,244,5,Sexta_W1,1
5875,Psicologia,inativa,_1605_PSICOLOGIA_INATIVA_0_18A25PT05_06112025_...,2025-11-07,19.0,64,0,Sexta_W1,1
5868,Medicina,inativa,_1605_MEDICINA_INATIVA_0_18A25PT01_07112025_V498,2025-11-07,15.0,54,2,Sexta_W1,1
...,...,...,...,...,...,...,...,...,...
656,Nutrição,inativa,_1605_NUTRICAO_INATIVA_0_31032026_V500_pt4,2026-04-07,11.0,272,5,Terça_W1,1
313,Fisioterapia,None,_1605__1605_FISIOTERAPIA_PAGEVIEW_06042026_V50...,2026-04-07,9.0,626,9,Terça_W1,1
8380,Fisioterapia,None,_1605__1605_FISIOTERAPIA_PAGEVIEW_06042026_V50...,2026-04-07,14.0,59,1,Terça_W1,1
3034,Fisioterapia,Base Leads,_1605_FISIOTERAPIA_BASELEADS_POS_GESTAOFISIOTE...,2026-04-07,14.0,199,6,Terça_W1,1


In [42]:
## performance bases | Hoje ## 
df_today = df[df['data'].dt.date == dt.datetime.today().date()]
df_today = df_today.groupby(['area','base_type']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_today['performance'] = (df_today['total_atendidas'] / df_today['Total_tentativas'] * 100).round(2)
df_today = df_today.sort_values('performance', ascending=False)
df_today

,area,base_type,Total_tentativas,total_atendidas,performance
8,Psicologia,ativa,1739,65,3.74
2,Fisioterapia,Base Leads,1022,34,3.33
3,Fisioterapia,inativa,1668,41,2.46
0,Enfermagem,Base Leads,1404,33,2.35
5,Multi,Base Leads,315,7,2.22
1,Enfermagem,inativa,1091,24,2.20
9,Psicologia,inativa,1035,21,2.03
7,Psicologia,Base Leads,51,1,1.96
6,Nutrição,inativa,1291,12,0.93
4,Medicina,Base Leads,345,3,0.87


In [43]:
## top10 bases ## 
df_top10 = df.groupby(['area','base_type']).agg(
    total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_top10['performance'] = (df_top10['total_atendidas'] / df_top10['total_tentativas'] * 100).round(2)
df_top10 = df_top10.sort_values('performance', ascending=False).head(10)
df_top10

,area,base_type,total_tentativas,total_atendidas,performance
57,Veterinária,ativa,58,6,10.34
25,Multi,Carrinho,66,5,7.58
55,Veterinária,Base Leads,4413,256,5.80
60,Veterinária,legado,167,9,5.39
31,Nutrição,Base Leads,5122,254,4.96
30,Multi,legado,64,3,4.69
28,Multi,evento,10897,467,4.29
26,Multi,Material Rico,3556,149,4.19
36,Pediatria,Base Leads,3153,131,4.15
40,Pediatria,evento,4805,189,3.93
